In [ ]:
!git clone https://github.com/BinaLab/RescueNet-A-High-Resolution-Post-Disaster-UAV-Dataset-for-Semantic-Segmentation.git

In [ ]:
!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/1jEiiIHkAeFeQ_ll48jbd9nkkdfkLIte7


In [ ]:
!cp -r RescueNet/* /content/drive/MyDrive/SharedRuns/RescueNet

In [ ]:
!wget -O RescueNet.zip "https://www.dropbox.com/scl/fo/ntgeyhxe2mzd2wuh7he7x/AFIchlfjVO_7MzPcNc1ZOHE/RescueNet?rlkey=6vxiaqve9gp6vzvzh3t5mz0vv&subfolder_nav_tracking=1&st=8884f7ne&dl=0"

In [ ]:
!unzip -q RescueNet.zip -d /content/RescueNet-A-High-Resolution-Post-Disaster-UAV-Dataset-for-Semantic-Segmentation-main/Segmentation-Experiments/dataset/RescueNet

In [ ]:
!pip install torch torchvision torchaudio tensorboardX pyyaml opencv-python pillow tqdm


In [ ]:
%cd /content

In [ ]:
!cp -vr exp/RescueNet/pspnet101/model /content/drive/MyDrive/RescueNet-run-1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall gdown

In [ ]:
!pip install -q gdown

!gdown --folder https://drive.google.com/drive/folders/1WvL9UHcde6QxgbgOnddt6l047T8rZfB4?usp=sharing

In [ ]:
!mv /content/SharedRuns/run1 /content/drive/MyDrive/SharedRuns

In [ ]:
%cd /content/RescueNet-A-High-Resolution-Post-Disaster-UAV-Dataset-for-Semantic-Segmentation-main/Segmentation-Experiments

In [ ]:
%cd /content

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/SharedRuns/run1'
!python train.py --config config/rescuenet-pspnet101.yaml \
    TRAIN.arch aunet \
    TRAIN.resume /content/drive/MyDrive/SharedRuns/run1/train_epoch_92.pth \
    TRAIN.save_path {SAVE_DIR} \
    TRAIN.save_freq 1

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/SharedRuns/run1'
!python test.py --config config/rescuenet-pspnet101.yaml \
    TEST.arch aunet \
    TEST.model_path {SAVE_DIR}/train_epoch_100.pth \
    TEST.mode test \
    TEST.imshow_batch False \
    TEST.save_folder {SAVE_DIR}/test_outputs


In [ ]:
import io, requests, torch
from PIL import Image
import torchvision.transforms as T
from models.unet import AttU_Net  # adjust to your arch
from util.dataset import color_encoding  # or load saved mapping
from util.util import colorize  # converts class map to RGB look-up table

# 1. Download and preprocess
url = "/content/test100.jpg"
response = requests.get(url)
img = Image.open(io.BytesIO(response.content)).convert("RGB")
transform = T.Compose([
    T.Resize((713, 713)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])
tensor = transform(img).unsqueeze(0).cuda()

# 2. Load trained checkpoint
ckpt = torch.load("/content/drive/MyDrive/SharedRuns/run1/train_epoch_100.pth")
model = AttU_Net(img_ch=3, output_ch=11)
model = torch.nn.DataParallel(model).cuda()
model.load_state_dict(ckpt["state_dict"])
model.eval()

# 3. Predict + colorize
with torch.no_grad():
    logits = model(tensor)
pred = logits.argmax(1).cpu().squeeze(0).numpy()  # H×W class IDs
color_mask = colorize(pred, color_encoding)        # H×W×3 RGB mask

# 4. Save overlay
Image.fromarray(color_mask).save("internet_image_mask.png")


In [ ]:
import torch
from PIL import Image
import torchvision.transforms as T

from models.unet import AttU_Net
from data.rescuenet import RescueNet as RescueNetDataset
from util.util import colorize

# 1) Build palette + preprocessing
color_encoding = list(RescueNetDataset.color_encoding.values())  # [(r,g,b), ...]
palette = []
for rgb in color_encoding:
    palette.extend(rgb)
# pad to 256*3 entries so PIL accepts it
palette.extend([0] * (256 * 3 - len(palette)))

transform = T.Compose([
    T.Resize((713, 713)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# 2) Load image
img = Image.open("/content/test100.jpg").convert("RGB")
tensor = transform(img).unsqueeze(0).cuda()

# 3) Load checkpoint (DataParallel weights)
ckpt = torch.load("/content/drive/MyDrive/SharedRuns/run1/train_epoch_100.pth", map_location="cuda")
model = AttU_Net(img_ch=3, output_ch=len(color_encoding))
model = torch.nn.DataParallel(model).cuda()
model.load_state_dict(ckpt["state_dict"])
model.eval()

# 4) Predict + colorize
with torch.no_grad():
    logits = model(tensor)
pred = logits.argmax(1).cpu().squeeze(0).numpy()

mask_img = colorize(pred, palette)
mask_img.save("internet_image_mask.png")
print("Saved prediction to internet_image_mask.png")



In [ ]:
ckpt_path = "/content/drive/MyDrive/SharedRuns/run1/train_epoch_100.pth"

In [ ]:
model = AttU_Net(img_ch=3, output_ch=11)
model = torch.nn.DataParallel(model)
state = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(state["state_dict"])


In [ ]:
from collections import OrderedDict
raw_state = torch.load(ckpt_path, map_location="cpu")["state_dict"]
clean_state = OrderedDict((k.replace("module.", "", 1), v) for k, v in raw_state.items())
model = AttU_Net(img_ch=3, output_ch=11)
model.load_state_dict(clean_state)


In [ ]:
import torch
from models.unet import AttU_Net  # or whichever arch you trained

ckpt_path = "/content/drive/MyDrive/SharedRuns/run1/train_epoch_100.pth"
onnx_path = "/content/drive/MyDrive/SharedRuns/attunet_rescuenet.onnx"

model = AttU_Net(img_ch=3, output_ch=11)
state = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(state["state_dict"])
model.eval()

dummy = torch.randn(1, 3, 713, 713)  # same input size as training

torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["input"], output_names=["logits"],
    opset_version=13, dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}
)


In [ ]:
!pip install onnx

In [ ]:
!pip install onnx2keras

In [ ]:
import onnx
from onnx2keras import onnx_to_keras
from tensorflow import keras

onnx_model = onnx.load("attunet_rescuenet.onnx")
keras_model = onnx_to_keras(
    onnx_model,
    input_names=["input"],
    change_ordering=True  # PyTorch is CHW; TF wants HWC
)

keras_model.save("attunet_rescuenet.keras")  # native TF format (TF 2.13+)
keras_model.save("attunet_rescuenet.h5")     # legacy HDF5


In [ ]:
!pip install onnxscript

In [ ]:
import torch, onnx, io
from collections import OrderedDict
from models.unet import AttU_Net
from onnx2keras import onnx_to_keras
from tensorflow import keras

# --- 1. Load checkpoint & clean keys ---
ckpt_path = "/content/drive/MyDrive/SharedRuns/run1/train_epoch_100.pth"
state = torch.load(ckpt_path, map_location="cpu")["state_dict"]
clean_state = OrderedDict((k.replace("module.", "", 1), v) for k, v in state.items())

model = AttU_Net(img_ch=3, output_ch=11)
model.load_state_dict(clean_state)
model.eval()

# --- 2. Export to ONNX ---
dummy = torch.randn(1, 3, 713, 713)
onnx_path = "/content/drive/MyDrive/SharedRuns/attunet_rescuenet.onnx"
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=13
)

# --- 3. Convert ONNX -> Keras (.keras + .h5) ---
onnx_model = onnx.load(onnx_path)
keras_model = onnx_to_keras(onnx_model, input_names=["input"], change_ordering=True)

keras_model.save("/content/drive/MyDrive/SharedRuns/attunet_rescuenet.keras")
keras_model.save("/content/drive/MyDrive/SharedRuns/attunet_rescuenet.h5")

print("Saved ONNX + Keras checkpoints.")


In [ ]:
!pip install "tensorflow==2.13.*" "keras==2.13.*" onnx2keras==0.0.24
# restart runtime


In [ ]:
!pip install timm


In [ ]:
import timm

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

data_root = Path("/content/RescueNet-A-High-Resolution-Post-Disaster-UAV-Dataset-for-Semantic-Segmentation/Segmentation-Experiments/dataset/RescueNet")
rgb_dir = data_root / "train" / "train-org-img"
mask_dir = data_root / "train" / "train-label-img"

samples = sorted(rgb_dir.glob("*.jpg"))[:4]   # pick any subset

plt.figure(figsize=(12, 8))
for idx, rgb_path in enumerate(samples):
    mask_path = mask_dir / (rgb_path.stem + "_lab.png")
    rgb = Image.open(rgb_path).convert("RGB")
    mask = Image.open(mask_path).convert("RGB")
    plt.subplot(len(samples), 2, 2*idx+1)
    plt.imshow(rgb)
    plt.axis("off")
    plt.title(rgb_path.name)
    plt.subplot(len(samples), 2, 2*idx+2)
    plt.imshow(mask)
    plt.axis("off")
    plt.title(mask_path.name)

plt.tight_layout()
plt.show()


In [ ]:
# Preview RGB images + colorized masks
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as T
import transforms as ext_transforms  # repo’s helper for mask coloring
from data.rescuenet import RescueNet

data_root = Path("/content/RescueNet-A-High-Resolution-Post-Disaster-UAV-Dataset-for-Semantic-Segmentation/Segmentation-Experiments/dataset/RescueNet")
rgb_dir = data_root / "train" / "train-org-img"
mask_dir = data_root / "train" / "train-label-img"

# pick a few samples; change the glob as needed
rgb_paths = sorted(rgb_dir.glob("*.jpg"))[:4]

# build the repo’s RGB palette transform once
label_to_rgb = T.Compose([
    ext_transforms.LongTensorToRGBPIL(RescueNet.color_encoding)
])

plt.figure(figsize=(12, 8))
for idx, rgb_path in enumerate(rgb_paths):
    mask_path = mask_dir / (rgb_path.stem + "_lab.png")
    rgb_img = Image.open(rgb_path).convert("RGB")

    mask_ids = Image.open(mask_path)
    mask_tensor = torch.from_numpy(np.array(mask_ids, dtype=np.int64))
    mask_rgb = label_to_rgb(mask_tensor)

    plt.subplot(len(rgb_paths), 2, 2*idx + 1)
    plt.imshow(rgb_img)
    plt.axis("off")
    plt.title(rgb_path.name)

    plt.subplot(len(rgb_paths), 2, 2*idx + 2)
    plt.imshow(mask_rgb)
    plt.axis("off")
    plt.title(mask_path.name)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as T
import transforms as ext_transforms
from data.rescuenet import RescueNet

data_root = Path("/content/RescueNet-A-High-Resolution-Post-Disaster-UAV-Dataset-for-Semantic-Segmentation/Segmentation-Experiments/dataset/RescueNet")
rgb_path = next(iter(sorted((data_root / "train" / "train-org-img").glob("*.jpg"))))
mask_path = (data_root / "train" / "train-label-img" / f"{rgb_path.stem}_lab.png")

label_to_rgb = T.Compose([ext_transforms.LongTensorToRGBPIL(RescueNet.color_encoding)])

rgb_img = Image.open(rgb_path).convert("RGB")
mask_tensor = torch.from_numpy(np.array(Image.open(mask_path), dtype=np.int64))
mask_rgb = label_to_rgb(mask_tensor)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(rgb_img)
plt.axis("off")
plt.title(rgb_path.name)

plt.subplot(1, 2, 2)
plt.imshow(mask_rgb)
plt.axis("off")
plt.title(mask_path.name)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as T

from models.unet import AttU_Net
from data.rescuenet import RescueNet
from util.util import colorize

# --- paths ---
img_path = "/content/flood.jpg"
ckpt_path = "/content/drive/MyDrive/SharedRuns/run1/train_epoch_100.pth"

# --- palette + preprocess ---
palette = []
for rgb in RescueNet.color_encoding.values():
    palette.extend(rgb)
palette.extend([0] * (256*3 - len(palette)))   # pad for PIL

transform = T.Compose([
    T.Resize((713, 713)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

rgb_img = Image.open(img_path).convert("RGB")
tensor = transform(rgb_img).unsqueeze(0).cuda()

# --- load model ---
state = torch.load(ckpt_path, map_location="cuda")["state_dict"]
model = AttU_Net(img_ch=3, output_ch=len(RescueNet.color_encoding))
model = torch.nn.DataParallel(model).cuda()
model.load_state_dict(state)
model.eval()

# --- predict ---
with torch.no_grad():
    logits = model(tensor)
pred = logits.argmax(1).cpu().squeeze(0).numpy()
mask_rgb = colorize(pred, palette)

# --- visualize ---
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(rgb_img)
plt.axis("off")
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(mask_rgb)
plt.axis("off")
plt.title("Predicted Segmentation")

plt.tight_layout()
plt.show()
